# ⚔️ DSWA: Nossa Primeira Competição Kaggle!

Bem-vindos ao nosso primeiro desafio prático! Hoje vamos participar da competição **Predict Customer Churn** no Kaggle.

**O Problema:** Uma empresa de telecomunicações quer saber quais clientes vão cancelar a assinatura (o que chamamos de *Churn*). Descobrir isso antes que o cliente vá embora permite que a empresa ofereça descontos ou vantagens para mantê-lo.

**Nosso Objetivo:** Ensinar uma Inteligência Artificial a olhar para o histórico dos clientes e prever a probabilidade de eles cancelarem o serviço!

Vamos fazer isso em 6 passos simples.

## Passo 1: Preparando a Caixa de Ferramentas 🧰

Antes de construir uma casa, precisamos de tijolos e cimento. No Python, usamos "bibliotecas", que são pacotes de códigos já prontos que nos ajudam a trabalhar com dados, criar gráficos e treinar a IA.

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

## Passo 2: Conhecendo Nossos Dados 🔍
Vamos carregar o arquivo train.csv (os dados dos clientes que sabemos se cancelaram ou não) e dar uma olhada neles.

In [6]:
PATH = '/kaggle/input/competitions/playground-series-s6e3/'
treino = pd.read_csv(PATH + 'train.csv')
teste = pd.read_csv(PATH + 'test.csv')

display(treino.head().style.hide(axis="index"))

id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.100000,1653.850000,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.500000,3778.200000,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.400000,5841.350000,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.700000,70.700000,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.450000,70.450000,Yes


## Passo 3: Traduzindo para a Máquina (Pré-processamento) 🤖
O computador é muito rápido, mas ele é "bobo": ele só entende números. Ele não sabe o que é "Fibra Ótica" ou "Feminino". Precisamos transformar todo o texto em números.

In [7]:
X = treino.drop(columns=['id', 'Churn'])
y = treino['Churn'].map({'No':0, 'Yes':1})

X_teste_comp = teste.drop(columns=['id'])

# Nova feature sugerida
X['cobr_mensal'] = X['MonthlyCharges'] / (X['tenure']+1)
X_teste_comp['cobr_mensal'] = X_teste_comp['MonthlyCharges'] / (X_teste_comp['tenure']+1)

X = pd.get_dummies(X, drop_first=True)
X_teste_comp = pd.get_dummies(X_teste_comp, drop_first=True)

X, X_teste_comp = X.align(X_teste_comp, join='inner', axis=1)

print("Dados traduzidos para números com sucesso!")
display(X.head())

Dados traduzidos para números com sucesso!


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,cobr_mensal,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,29,60.10,1653.85,2.003333,True,True,True,True,False,...,False,False,False,False,True,False,True,False,False,True
1,0,58,69.50,3778.20,1.177966,True,True,True,True,False,...,False,True,False,False,False,True,False,True,False,False
2,0,58,100.40,5841.35,1.701695,True,True,False,True,False,...,False,True,False,True,False,False,True,False,True,False
3,0,1,69.70,70.70,34.850000,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
4,0,1,70.45,70.45,35.225000,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


## Passo 4: O Treinamento (Machine Learning) 🧠
Antes de colocar nosso modelo para fazer a prova do Kaggle, precisamos testá-lo em casa.
Para isso, vamos separar nossos dados de treino: 80% para a IA estudar, e 20% para a gente fazer um simulado.

Por que separamos dados para o simulado? Porque se a IA estudar todas as questões, ela pode simplesmente 'decorar' as respostas em vez de aprender a lógica. O simulado com questões que ela nunca viu garante que ela realmente aprendeu a matéria!

Neste notebook, vamos usar o **Random Forest** (Floresta Aleatória). Imagine que, em vez de perguntar para um único especialista se o cliente vai cancelar, nós perguntamos para uma equipe de 100 especialistas diferentes e tiramos uma média da opinião deles. É isso que esse algoritmo faz!

In [8]:
X_treino, X_valid, y_treino, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    eval_metric='auc',
    random_state=42
)
modelo.fit(X_treino, y_treino)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

## Passo 5: Avaliação do Modelo 📊
O Kaggle vai nos avaliar usando uma métrica chamada **ROC AUC**.

Pense assim: a IA dá uma nota de 0 a 100 para o risco de cada cliente cancelar. O ROC AUC avalia se a IA deu as notas mais altas realmente para as pessoas que cancelaram.
- **Nota 0.50:** A IA está apenas "chutando" (jogando uma moeda).
- **Nota 1.00:** A IA acertou perfeitamente.

In [9]:
previsoes_simulado = modelo.predict_proba(X_valid)[:, 1]

nota_auc = roc_auc_score(y_valid, previsoes_simulado)
print("="*40)
print(f"🎉 NOSSA NOTA NO SIMULADO: {nota_auc:.4f} 🎉")
print("="*40)

🎉 NOSSA NOTA NO SIMULADO: 0.9153 🎉


Lembra dos 100 especialistas do Random Forest? Aqui está o relatório do que eles mais olharam na ficha do cliente para tomar uma decisão. Geralmente, o tempo de contrato (tenure) e o valor da mensalidade e anuidade são os fatores decisivos!

## Passo 6: Gerando o Arquivo para o Kaggle! 🚀

Agora que vimos que o modelo aprendeu alguma coisa no simulado, vamos usá-lo para prever o arquivo ```test.csv``` oficial do Kaggle.

In [10]:
previsoes_finais = modelo.predict_proba(X_teste_comp)[:, 1]

submissao = pd.DataFrame({
    'id': teste['id'],
    'Churn': previsoes_finais
})

submissao.to_csv('submission.csv', index=False)
print("Arquivo salvo! Agora é só enviar no site do Kaggle!")
# Para submeter, vá na aba 'Submit to competition' na direita e clique no botão 'Submit'

Arquivo salvo! Agora é só enviar no site do Kaggle!


## 💡 Desafio para os Membros: Como subir no Ranking?

Esse notebook é apenas o nosso ponto de partida (nosso "Baseline"). A nota que tiramos não é a máxima possível. O verdadeiro trabalho de um Cientista de Dados começa agora!

Aqui estão algumas estratégias legais (e simples) que vocês podem tentar sozinhos para melhorar o modelo e conseguir uma pontuação maior:

1. **Feature Engineering (Engenharia de Variáveis):** Às vezes, criar uma nova coluna ajuda o modelo. Por exemplo, criar uma coluna "Cobrança Mensal por Mês de Contrato" dividindo as taxas pelo tempo de uso (```tenure```).

2. **Brincar com os Hiperparâmetros:** No ```RandomForestClassifier(n_estimators=100)```, o que acontece se você mudar para ```n_estimators=500```? Ou se limitar a profundidade da árvore de decisão com ```max_depth=5```? O modelo melhora ou piora?

3. **Mudar a IA:** Nós usamos Random Forest. Que tal importar e testar o ```LogisticRegression``` ou o ```XGBoost```? Algoritmos diferentes aprendem de jeitos diferentes.

4. **Balanceando o modelo:** Como temos muito mais dados de quem não cancelou, sua IA pode ter dificuldade em aprender os padrões de quem sai. Que tal testar o parâmetro ```class_weight='balanced'``` para forçar o modelo a dar a mesma importância aos dois grupos?

*Escolha uma dessas ideias, altere o código acima e veja se sua nota sobe! Boa sorte!*